# FT-Transformer: Single Model Performance on cHL CODEX
## Baseline: Ensemble achieves 90.27% F1
## Goal: Test FT-Transformer alone to understand its contribution to ensemble performance

## 1. Import Required Libraries

In [ ]:
import os, random, time, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, SequentialSampler

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report, confusion_matrix, accuracy_score

print(f"PyTorch: {torch.__version__}")
print(f"Device  : {'GPU (' + torch.cuda.get_device_name(0) + ')' if torch.cuda.is_available() else 'CPU'}")

SEED = 7325111
def set_seed(seed: int):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 2. Load and Explore the Dataset

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/amshahriarrashidmahe/chl-codex-annotated/cHL_CODEX_annotation.csv")

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nColumn names:")
print(df.columns.tolist())
print(f"\nData types:")
print(df.dtypes)
print(f"\nMissing values:")
print(df.isnull().sum())
print(f"\nClass distribution:")
print(df['cellType'].value_counts())

## 3. Preprocess the Data

In [ ]:
# Define marker columns and label mapping
MARKER_COLS = [
    'BCL.2','CCR6','CD11b','CD11c','CD15','CD16','CD162','CD163',
    'CD2','CD20','CD206','CD25','CD30','CD31','CD4','CD44',
    'CD45','CD45RA','CD45RO','CD5','CD56','CD57','CD68','CD69',
    'CD7','CD8','Collagen.4','Cytokeratin','DAPI.01','EGFR',
    'FoxP3','Granzyme.B','HLA.DR','IDO.1','LAG.3','MCT','MMP.9',
    'MUC.1','PD.1','PD.L1','Podoplanin','T.bet','TCR.g.d','TCRb',
    'Tim.3','VISA','Vimentin','a.SMA','b.Catenin',
]
FEATURE_COLS = MARKER_COLS + ['cellSize']
CLASS_NAMES = ['B','CD4','CD8','DC','Endothelial','Epithelial',
               'Lymphatic','M1','M2','Mast','Monocyte','NK',
               'Neutrophil','Other','TReg','Tumor']
LABEL_MAP = {n: i for i, n in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)
NUM_FEATURES = len(FEATURE_COLS)

print(f"Number of features: {NUM_FEATURES}")
print(f"Number of classes: {NUM_CLASSES}")

# Filter valid classes and create feature/label arrays
df = df[df['cellType'].isin(CLASS_NAMES)].reset_index(drop=True)
X_all = df[FEATURE_COLS].values.astype(np.float64)
y_all = df['cellType'].map(LABEL_MAP).values.astype(np.int64)

# Train-test split
X_train, X_valid, y_train, y_valid = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)

# Z-score normalization
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train).astype(np.float32)
X_valid_sc = scaler.transform(X_valid).astype(np.float32)

print(f"Training set size: {len(X_train):,}")
print(f"Validation set size: {len(X_valid):,}")

## 4. Dataset Class and Data Loaders

In [ ]:
class CellDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.X[i], self.y[i]

def make_loaders(X_tr, y_tr, X_va, y_va, batch_size=256):
    ds_tr = CellDataset(X_tr, y_tr)
    ds_va = CellDataset(X_va, y_va)

    counts = np.bincount(y_tr)
    weights = 1.0 / counts[y_tr]
    sampler = WeightedRandomSampler(weights, len(weights), replacement=True)

    tr_loader = DataLoader(ds_tr, batch_size=batch_size, sampler=sampler, drop_last=True)
    va_loader = DataLoader(ds_va, batch_size=batch_size, sampler=SequentialSampler(ds_va), drop_last=False)
    return tr_loader, va_loader

tr_loader, va_loader = make_loaders(X_train_sc, y_train, X_valid_sc, y_valid, batch_size=256)
print(f"Training batches: {len(tr_loader)}")
print(f"Validation batches: {len(va_loader)}")

## 5. Implement FT-Transformer Architecture

In [ ]:
class FeatureTokenizer(nn.Module):
    """Converts each feature into a learned token embedding"""
    def __init__(self, n_features: int, d_token: int):
        super().__init__()
        self.W = nn.Parameter(torch.empty(n_features, d_token))
        self.b = nn.Parameter(torch.zeros(n_features, d_token))
        nn.init.kaiming_uniform_(self.W, a=math.sqrt(5))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x.unsqueeze(-1) * self.W + self.b

class FTTransformer(nn.Module):
    """Feature-Tokenizer Transformer for tabular data"""
    def __init__(self, n_features=50, d_token=64, n_heads=8, n_layers=3, 
                 ffn_mult=4, dropout=0.1, n_classes=16):
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_features, d_token)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_token))
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_token, nhead=n_heads, 
            dim_feedforward=d_token*ffn_mult,
            dropout=dropout, batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head = nn.Sequential(nn.LayerNorm(d_token), nn.Linear(d_token, n_classes))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B = x.size(0)
        tokens = self.tokenizer(x)
        cls = self.cls_token.expand(B, -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)
        out = self.encoder(tokens)
        return self.head(out[:, 0])

model = FTTransformer(n_features=NUM_FEATURES, d_token=64, n_heads=8, n_layers=3).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"FT-Transformer initialized with {total_params:,} parameters")

## 6. Configure and Train the Model

In [ ]:
def train_ft_transformer(model, tr_loader, va_loader, max_epochs=200, patience=50, min_epochs=100):
    optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)
    criterion = nn.CrossEntropyLoss()

    best_loss = float('inf')
    best_state = None
    patience_ctr = 0
    train_losses = []
    val_losses = []

    print(f"Training FT-Transformer for max {max_epochs} epochs...")
    for epoch in range(max_epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        for X_b, y_b in tr_loader:
            optimizer.zero_grad()
            logits = model(X_b.to(DEVICE))
            loss = criterion(logits, y_b.to(DEVICE))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item() * len(y_b)
        
        train_loss /= len(y_train)
        train_losses.append(train_loss)
        scheduler.step()

        # Validation phase
        model.eval()
        va_loss = 0.0
        with torch.no_grad():
            for X_b, y_b in va_loader:
                logits = model(X_b.to(DEVICE))
                loss = criterion(logits, y_b.to(DEVICE))
                va_loss += loss.item() * len(y_b)
        
        va_loss /= len(y_valid)
        val_losses.append(va_loss)

        if va_loss < best_loss:
            best_loss = va_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
            if epoch % 20 == 0:
                print(f"Epoch {epoch:3d} | train_loss={train_loss:.4f} | val_loss={va_loss:.4f} ← best")
        else:
            patience_ctr += 1

        if patience_ctr > patience and epoch >= min_epochs:
            print(f"Early stopping at epoch {epoch}")
            break

    model.load_state_dict(best_state)
    return model, train_losses, val_losses

set_seed(SEED)
model = FTTransformer(n_features=NUM_FEATURES, d_token=64, n_heads=8, n_layers=3).to(DEVICE)
model, train_losses, val_losses = train_ft_transformer(model, tr_loader, va_loader, max_epochs=200, patience=50)

## 7. Evaluate Model Performance

In [ ]:
@torch.no_grad()
def predict(model, X_sc):
    model.eval()
    ds = CellDataset(X_sc, np.zeros(len(X_sc)))
    loader = DataLoader(ds, batch_size=512, shuffle=False)
    all_probs = []
    for X_b, _ in loader:
        logits = model(X_b.to(DEVICE))
        probs = torch.softmax(logits, dim=-1)
        all_probs.append(probs.cpu().numpy())
    return np.concatenate(all_probs, axis=0)

ft_probs = predict(model, X_valid_sc)
ft_preds = ft_probs.argmax(axis=1)

ft_acc = accuracy_score(y_valid, ft_preds)
ft_f1 = f1_score(y_valid, ft_preds, average='weighted')
ft_f1_macro = f1_score(y_valid, ft_preds, average='macro')

print("="*60)
print("FT-TRANSFORMER PERFORMANCE")
print("="*60)
print(f"Accuracy    : {ft_acc:.4f}")
print(f"Weighted F1 : {ft_f1:.4f}")
print(f"Macro F1    : {ft_f1_macro:.4f}")
print("="*60)
print("\nComparison to Baseline:")
print(f"  Ensemble (FYDP-II) : 90.27% F1")
print(f"  FT-Transformer     : {ft_f1*100:.2f}% F1")
print(f"  Difference         : {(ft_f1 - 0.9027)*100:+.2f}pp")
print("="*60)

In [ ]:
print("\nDetailed Classification Report:")
print(classification_report(y_valid, ft_preds, target_names=CLASS_NAMES, digits=4))

## 8. Visualize Results and Metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, label='Training Loss', linewidth=2)
axes[0].plot(val_losses, label='Validation Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('FT-Transformer Training Curves')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

f1_per_class = []
for i, class_name in enumerate(CLASS_NAMES):
    mask = (y_valid == i)
    if mask.sum() > 0:
        class_f1 = f1_score(y_valid[mask], ft_preds[mask], average='weighted')
        f1_per_class.append(class_f1)
    else:
        f1_per_class.append(0)

sorted_idx = np.argsort(f1_per_class)
axes[1].barh(range(len(CLASS_NAMES)), [f1_per_class[i] for i in sorted_idx], color='#42A5F5')
axes[1].set_yticks(range(len(CLASS_NAMES)))
axes[1].set_yticklabels([CLASS_NAMES[i] for i in sorted_idx])
axes[1].set_xlabel('F1 Score')
axes[1].set_title('Per-Class F1 Scores (Sorted)')
axes[1].set_xlim([0, 1])
axes[1].axvline(ft_f1, color='red', linestyle='--', label=f'Weighted F1: {ft_f1:.4f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('ft_transformer_training_performance.png', dpi=150)
plt.show()

print("Training performance visualization saved.")

In [ ]:
cm = confusion_matrix(y_valid, ft_preds)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix - FT-Transformer')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('ft_transformer_confusion_matrix.png', dpi=150)
plt.show()

print("Confusion matrix saved.")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

models = ['LightGBM\n(FYDP-II)', 'XGBoost\n(FYDP-II)', 'MLP\n(FYDP-II)', 
          'Ensemble\n(FYDP-II)', 'FT-Transformer\n(Current)']
scores = [0.8868, 0.8765, 0.8701, 0.9027, ft_f1]
colors = ['#66BB6A', '#66BB6A', '#FFA726', '#42A5F5', '#AB47BC']

bars = ax.bar(models, scores, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)

for bar, score in zip(bars, scores):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{score*100:.2f}%', ha='center', va='bottom', fontweight='bold')

ax.axhline(0.90, color='red', linestyle='--', linewidth=2, label='90% Target')
ax.set_ylabel('Weighted F1 Score', fontsize=12, fontweight='bold')
ax.set_ylim([0.80, 0.95])
ax.set_title('FT-Transformer vs. FYDP-II Baseline Models', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('ft_transformer_vs_baseline.png', dpi=150)
plt.show()

print("Model comparison visualization saved.")

## Summary and Insights

### Key Findings:
- **FT-Transformer Performance**: See results above
- **Comparison to Ensemble**: Difference relative to 90.27%
- **Interpretation**: The FT-Transformer is designed for interpretability, not maximal accuracy. Adding it as a diverse ensemble member may improve overall performance through model diversity.

### Next Steps:
1. **Ensemble Integration**: Add FT-Transformer predictions to the existing MLP+LightGBM+XGBoost+CatBoost ensemble
2. **Weight Optimization**: Grid search for optimal ensemble weights including FT-Transformer
3. **Attention Analysis**: Extract attention heatmaps to understand which markers the model focuses on per cell type

## FT-Transformer with cHL-CODEX

### Key Findings:
- **FT-Transformer Performance**: {ft_f1*100:.2f}% weighted F1
- **Comparison to Ensemble**: {(ft_f1 - 0.9027)*100:+.2f}pp relative to 90.27%
- **Interpretation**: The FT-Transformer is designed for interpretability, not maximal accuracy. Adding it as a diverse ensemble member may improve overall performance through model diversity.


In [1]:
fig, ax = plt.subplots(figsize=(10, 6))

models = ['LightGBM\n(FYDP-II)', 'XGBoost\n(FYDP-II)', 'MLP\n(FYDP-II)', 
          'Ensemble\n(FYDP-II)', 'FT-Transformer\n(Current)']
scores = [0.8868, 0.8765, 0.8701, 0.9027, ft_f1]
colors = ['#66BB6A', '#66BB6A', '#FFA726', '#42A5F5', '#AB47BC']

bars = ax.bar(models, scores, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)

# Add value labels on bars
for bar, score in zip(bars, scores):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{score*100:.2f}%', ha='center', va='bottom', fontweight='bold')

ax.axhline(0.90, color='red', linestyle='--', linewidth=2, label='90% Target')
ax.set_ylabel('Weighted F1 Score', fontsize=12, fontweight='bold')
ax.set_ylim([0.80, 0.95])
ax.set_title('FT-Transformer vs. FYDP-II Baseline Models', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('ft_transformer_vs_baseline.png', dpi=150)
plt.show()

print("Model comparison visualization saved.")

NameError: name 'plt' is not defined

In [ ]:
cm = confusion_matrix(y_valid, ft_preds)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix - FT-Transformer')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('ft_transformer_confusion_matrix.png', dpi=150)
plt.show()

print("Confusion matrix saved.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training curves
axes[0].plot(train_losses, label='Training Loss', linewidth=2)
axes[0].plot(val_losses, label='Validation Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('FT-Transformer Training Curves')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Per-class F1 scores
f1_per_class = []
for i, class_name in enumerate(CLASS_NAMES):
    mask = (y_valid == i)
    if mask.sum() > 0:
        class_f1 = f1_score(y_valid[mask], ft_preds[mask], average='weighted')
        f1_per_class.append(class_f1)
    else:
        f1_per_class.append(0)

sorted_idx = np.argsort(f1_per_class)
axes[1].barh(range(len(CLASS_NAMES)), [f1_per_class[i] for i in sorted_idx], color='#42A5F5')
axes[1].set_yticks(range(len(CLASS_NAMES)))
axes[1].set_yticklabels([CLASS_NAMES[i] for i in sorted_idx])
axes[1].set_xlabel('F1 Score')
axes[1].set_title('Per-Class F1 Scores (Sorted)')
axes[1].set_xlim([0, 1])
axes[1].axvline(ft_f1, color='red', linestyle='--', label=f'Weighted F1: {ft_f1:.4f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('ft_transformer_training_performance.png', dpi=150)
plt.show()

print("Training performance visualization saved.")

## 8. Visualize Results and Metrics

In [ ]:
print("\nDetailed Classification Report:")
print(classification_report(y_valid, ft_preds, target_names=CLASS_NAMES, digits=4))

In [ ]:
@torch.no_grad()
def predict(model, X_sc):
    model.eval()
    ds = CellDataset(X_sc, np.zeros(len(X_sc)))
    loader = DataLoader(ds, batch_size=512, shuffle=False)
    all_probs = []
    for X_b, _ in loader:
        logits = model(X_b.to(DEVICE))
        probs = torch.softmax(logits, dim=-1)
        all_probs.append(probs.cpu().numpy())
    return np.concatenate(all_probs, axis=0)

# Get predictions
ft_probs = predict(model, X_valid_sc)
ft_preds = ft_probs.argmax(axis=1)

# Calculate metrics
ft_acc = accuracy_score(y_valid, ft_preds)
ft_f1 = f1_score(y_valid, ft_preds, average='weighted')
ft_f1_macro = f1_score(y_valid, ft_preds, average='macro')

print("="*60)
print("FT-TRANSFORMER PERFORMANCE")
print("="*60)
print(f"Accuracy    : {ft_acc:.4f}")
print(f"Weighted F1 : {ft_f1:.4f}")
print(f"Macro F1    : {ft_f1_macro:.4f}")
print("="*60)
print("\nComparison to Baseline:")
print(f"  Ensemble (FYDP-II) : 90.27% F1")
print(f"  FT-Transformer     : {ft_f1*100:.2f}% F1")
print(f"  Difference         : {(ft_f1 - 0.9027)*100:+.2f}pp")
print("="*60)

## 7. Evaluate Model Performance

In [ ]:
def train_ft_transformer(model, tr_loader, va_loader, max_epochs=200, patience=50, min_epochs=100):
    optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)
    criterion = nn.CrossEntropyLoss()

    best_loss = float('inf')
    best_state = None
    patience_ctr = 0
    train_losses = []
    val_losses = []

    print(f"Training FT-Transformer for max {max_epochs} epochs...")
    for epoch in range(max_epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        for X_b, y_b in tr_loader:
            optimizer.zero_grad()
            logits = model(X_b.to(DEVICE))
            loss = criterion(logits, y_b.to(DEVICE))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item() * len(y_b)
        
        train_loss /= len(y_train)
        train_losses.append(train_loss)
        scheduler.step()

        # Validation phase
        model.eval()
        va_loss = 0.0
        with torch.no_grad():
            for X_b, y_b in va_loader:
                logits = model(X_b.to(DEVICE))
                loss = criterion(logits, y_b.to(DEVICE))
                va_loss += loss.item() * len(y_b)
        
        va_loss /= len(y_valid)
        val_losses.append(va_loss)

        # Early stopping logic
        if va_loss < best_loss:
            best_loss = va_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
            if epoch % 20 == 0:
                print(f"Epoch {epoch:3d} | train_loss={train_loss:.4f} | val_loss={va_loss:.4f} ← best")
        else:
            patience_ctr += 1

        if patience_ctr > patience and epoch >= min_epochs:
            print(f"Early stopping at epoch {epoch}")
            break

    model.load_state_dict(best_state)
    return model, train_losses, val_losses

# Train the model
set_seed(SEED)
model = FTTransformer(n_features=NUM_FEATURES, d_token=64, n_heads=8, n_layers=3).to(DEVICE)
model, train_losses, val_losses = train_ft_transformer(model, tr_loader, va_loader, max_epochs=200, patience=50)

## 6. Configure and Train the Model

In [ ]:
class FeatureTokenizer(nn.Module):
    """Converts each feature into a learned token embedding"""
    def __init__(self, n_features: int, d_token: int):
        super().__init__()
        self.W = nn.Parameter(torch.empty(n_features, d_token))
        self.b = nn.Parameter(torch.zeros(n_features, d_token))
        nn.init.kaiming_uniform_(self.W, a=math.sqrt(5))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x shape: (batch_size, n_features)
        # output shape: (batch_size, n_features, d_token)
        return x.unsqueeze(-1) * self.W + self.b

class FTTransformer(nn.Module):
    """Feature-Tokenizer Transformer for tabular data"""
    def __init__(self, n_features=50, d_token=64, n_heads=8, n_layers=3, 
                 ffn_mult=4, dropout=0.1, n_classes=16):
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_features, d_token)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_token))
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_token, nhead=n_heads, 
            dim_feedforward=d_token*ffn_mult,
            dropout=dropout, batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head = nn.Sequential(nn.LayerNorm(d_token), nn.Linear(d_token, n_classes))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B = x.size(0)
        tokens = self.tokenizer(x)  # (B, F, d_token)
        cls = self.cls_token.expand(B, -1, -1)  # (B, 1, d_token)
        tokens = torch.cat([cls, tokens], dim=1)  # (B, F+1, d_token)
        out = self.encoder(tokens)  # (B, F+1, d_token)
        return self.head(out[:, 0])  # Use CLS token for classification

# Test the model
model = FTTransformer(n_features=NUM_FEATURES, d_token=64, n_heads=8, n_layers=3).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"FT-Transformer initialized with {total_params:,} parameters")

## 5. Implement FT-Transformer Architecture

In [ ]:
class CellDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.X[i], self.y[i]

def make_loaders(X_tr, y_tr, X_va, y_va, batch_size=256):
    ds_tr = CellDataset(X_tr, y_tr)
    ds_va = CellDataset(X_va, y_va)

    counts = np.bincount(y_tr)
    weights = 1.0 / counts[y_tr]
    sampler = WeightedRandomSampler(weights, len(weights), replacement=True)

    tr_loader = DataLoader(ds_tr, batch_size=batch_size, sampler=sampler, drop_last=True)
    va_loader = DataLoader(ds_va, batch_size=batch_size, sampler=SequentialSampler(ds_va), drop_last=False)
    return tr_loader, va_loader

tr_loader, va_loader = make_loaders(X_train_sc, y_train, X_valid_sc, y_valid, batch_size=256)
print(f"Training batches: {len(tr_loader)}")
print(f"Validation batches: {len(va_loader)}")

## 4. Dataset Class and Data Loaders

In [ ]:
# Define marker columns and label mapping
MARKER_COLS = [
    'BCL.2','CCR6','CD11b','CD11c','CD15','CD16','CD162','CD163',
    'CD2','CD20','CD206','CD25','CD30','CD31','CD4','CD44',
    'CD45','CD45RA','CD45RO','CD5','CD56','CD57','CD68','CD69',
    'CD7','CD8','Collagen.4','Cytokeratin','DAPI.01','EGFR',
    'FoxP3','Granzyme.B','HLA.DR','IDO.1','LAG.3','MCT','MMP.9',
    'MUC.1','PD.1','PD.L1','Podoplanin','T.bet','TCR.g.d','TCRb',
    'Tim.3','VISA','Vimentin','a.SMA','b.Catenin',
]
FEATURE_COLS = MARKER_COLS + ['cellSize']  # 50 features
CLASS_NAMES = ['B','CD4','CD8','DC','Endothelial','Epithelial',
               'Lymphatic','M1','M2','Mast','Monocyte','NK',
               'Neutrophil','Other','TReg','Tumor']
LABEL_MAP = {n: i for i, n in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)
NUM_FEATURES = len(FEATURE_COLS)

print(f"Number of features: {NUM_FEATURES}")
print(f"Number of classes: {NUM_CLASSES}")

# Filter valid classes and create feature/label arrays
df = df[df['cellType'].isin(CLASS_NAMES)].reset_index(drop=True)
X_all = df[FEATURE_COLS].values.astype(np.float64)
y_all = df['cellType'].map(LABEL_MAP).values.astype(np.int64)

# Train-test split
X_train, X_valid, y_train, y_valid = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)

# Z-score normalization
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train).astype(np.float32)
X_valid_sc = scaler.transform(X_valid).astype(np.float32)

print(f"Training set size: {len(X_train):,}")
print(f"Validation set size: {len(X_valid):,}")

## 3. Preprocess the Data

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/amshahriarrashidmahe/chl-codex-annotated/cHL_CODEX_annotation.csv")

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nColumn names:")
print(df.columns.tolist())
print(f"\nData types:")
print(df.dtypes)
print(f"\nMissing values:")
print(df.isnull().sum())
print(f"\nClass distribution:")
print(df['cellType'].value_counts())

## 2. Load and Explore the Dataset

In [ ]:
import os, random, time, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, SequentialSampler

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report, confusion_matrix, accuracy_score

print(f"PyTorch: {torch.__version__}")
print(f"Device  : {'GPU (' + torch.cuda.get_device_name(0) + ')' if torch.cuda.is_available() else 'CPU'}")

SEED = 7325111
def set_seed(seed: int):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Import Required Libraries

# FT-Transformer: Single Model Performance on cHL CODEX
## Baseline: Ensemble achieves 90.27% F1
## Goal: Test FT-Transformer alone to understand its contribution to ensemble performance